# Vélib’ Métropole — Kit Data Science

**DATA-IA-EXE710 · Télécom Paris · Projet de groupe**

**Question :** une station Vélib’ sera-t-elle vide dans 30 minutes ?

Ce notebook réalise le contrôle qualité, la préparation et une première exploration d'un historique constitué par notre propre collecteur GBFS. Aucun historique fictif n'est inclus. Le code fonctionne quand au moins un instantané a été enregistré dans `data/raw/`.


In [ ]:
# 1. Importer le projet GitHub sur Google Colab (dépôt public).
import subprocess
from pathlib import Path

GITHUB_OWNER = "G-Akram01"  # À modifier si le dépôt est créé ailleurs
REPO = "velib-kit-data-science"
REPO_URL = f"https://github.com/{GITHUB_OWNER}/{REPO}.git"
ROOT = Path("/content") / REPO if Path("/content").exists() else Path.cwd()
if not (ROOT / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=True)
print("Projet chargé :", ROOT)


In [ ]:
# 2. Dépendances et lecture des fichiers JSON gzip.
import gzip
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

files = sorted((ROOT / "data" / "raw").glob("*/*.json.gz"))
print("Nombre d'instantanés :", len(files))
if not files:
    raise FileNotFoundError("Pas de données historiques : lancez d'abord le workflow GitHub Actions.")

def read_gzip(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)

def to_frame(path):
    payload = read_gzip(path)
    frame = pd.json_normalize(payload["stations"])
    frame["snapshot_at_utc"] = pd.to_datetime(payload["collected_at_utc"], utc=True)
    frame["source_last_updated"] = payload.get("source_last_updated")
    return frame

df = pd.concat([to_frame(p) for p in files], ignore_index=True)
df["station_id"] = df["station_id"].astype(str)
print("Dimensions brutes :", df.shape)
display(df.head())


In [ ]:
# 3. Nettoyage, typage et fusion éventuelle avec le référentiel stations.
def coalesce(frame, names):
    columns = [frame[name] for name in names if name in frame.columns]
    if not columns:
        return pd.Series(np.nan, index=frame.index)
    out = columns[0].copy()
    for series in columns[1:]:
        out = out.fillna(series)
    return out

for output, candidates in {
    "bikes_available": ["num_bikes_available", "numBikesAvailable", "num_vehicles_available"],
    "docks_available": ["num_docks_available", "numDocksAvailable"],
}.items():
    df[output] = pd.to_numeric(coalesce(df, candidates), errors="coerce")

for flag in ["is_installed", "is_renting", "is_returning"]:
    if flag in df:
        df[flag] = df[flag].map({True: 1, False: 0, 1: 1, 0: 0, "1": 1, "0": 0,
                                  "OUI": 1, "NON": 0, "true": 1, "false": 0})

if "last_reported" in df:
    df["last_reported_utc"] = pd.to_datetime(df["last_reported"], unit="s", utc=True, errors="coerce")
    df["source_age_minutes"] = (df["snapshot_at_utc"] - df["last_reported_utc"]).dt.total_seconds() / 60

ref_files = sorted((ROOT / "data" / "reference").glob("station_information_*.json.gz"))
if ref_files:
    reference = pd.json_normalize(read_gzip(ref_files[-1])["stations"])
    reference["station_id"] = reference["station_id"].astype(str)
    wanted = [c for c in ["station_id", "name", "lat", "lon", "capacity"] if c in reference]
    df = df.merge(reference[wanted].drop_duplicates("station_id"), on="station_id", how="left", validate="many_to_one")
else:
    print("Pas encore de référentiel station_information : analyse sans géolocalisation.")

df = df.drop_duplicates(["station_id", "snapshot_at_utc"], keep="last")
df = df.sort_values(["station_id", "snapshot_at_utc"]).reset_index(drop=True)
print("Dimensions nettoyées :", df.shape)
display(df.head())


In [ ]:
# Audit des 48 heures de collecte (sans données simulées).
import sys
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.audit_collection import audit
audit_report = audit(ROOT / 'data', hours=48)
print(json.dumps(audit_report, ensure_ascii=False, indent=2))
if not audit_report['complete_48h']:
    print('La fenêtre de collecte de 48 h n’est pas encore entièrement validée.')


In [ ]:
# 4. Qualité des données et couverture temporelle.
quality = pd.DataFrame({"colonne": df.columns, "type": df.dtypes.astype(str).values,
                        "manquants": df.isna().sum().values})
quality["% manquants"] = (quality["manquants"] / len(df) * 100).round(2)
display(quality.sort_values("% manquants", ascending=False))

snapshots = df["snapshot_at_utc"].nunique()
print("Stations uniques :", df["station_id"].nunique())
print("Instantanés uniques :", snapshots)
print("Début :", df["snapshot_at_utc"].min())
print("Fin :", df["snapshot_at_utc"].max())

unique_times = pd.Series(sorted(df["snapshot_at_utc"].unique()))
if len(unique_times) > 1:
    intervals = unique_times.diff().dropna().dt.total_seconds() / 60
    print("Intervalle médian réel (minutes) :", round(intervals.median(), 2))
    print("Intervalle maximal observé (minutes) :", round(intervals.max(), 2))

if "is_renting" in df:
    print("Valeurs is_renting :", df["is_renting"].value_counts(dropna=False).to_dict())


In [ ]:
# 5. Exploration : évolution du nombre total de vélos disponibles.
by_snapshot = df.groupby("snapshot_at_utc")["bikes_available"].sum(min_count=1)
ax = by_snapshot.plot(figsize=(12, 4), marker=".", title="Vélos disponibles - total des stations")
ax.set_xlabel("Horodatage UTC")
ax.set_ylabel("Nombre de vélos")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Part des stations ouvertes qui sont vides (et non les stations hors service).
active = df.loc[df["is_renting"] == 1].copy() if "is_renting" in df else df.copy()
active["empty_now"] = active["bikes_available"].eq(0)
active["hour_paris"] = active["snapshot_at_utc"].dt.tz_convert("Europe/Paris").dt.hour
hourly = active.groupby("hour_paris")["empty_now"].mean().mul(100)
ax = hourly.plot(kind="bar", figsize=(10, 4), title="Stations vides par heure locale (échantillon collecté)")
ax.set_ylabel("Stations vides (%)")
ax.set_xlabel("Heure (Europe/Paris)")
plt.tight_layout()
plt.show()


In [ ]:
# 6. Cible exploratoire : une station sera-t-elle vide dans ~30 minutes ?
# Recherche de la mesure la plus proche de T+30min, avec tolérance +/-3min.
# Ne pas faire une évaluation aléatoire : la séparation train/test doit être chronologique.

base = active.dropna(subset=["bikes_available"])[
    ["station_id", "snapshot_at_utc", "bikes_available", "empty_now"]
].copy()
base["future_query_utc"] = base["snapshot_at_utc"] + pd.Timedelta(minutes=30)
future = active.dropna(subset=["bikes_available"])[
    ["station_id", "snapshot_at_utc", "bikes_available"]
].rename(columns={"snapshot_at_utc": "future_snapshot_utc", "bikes_available": "future_bikes"})

pairs = pd.merge_asof(
    base.sort_values("future_query_utc"),
    future.sort_values("future_snapshot_utc"),
    left_on="future_query_utc", right_on="future_snapshot_utc",
    by="station_id", direction="nearest", tolerance=pd.Timedelta(minutes=3)
)
labeled = pairs.dropna(subset=["future_bikes"]).copy()
if len(labeled):
    labeled["empty_in_30min"] = labeled["future_bikes"].eq(0)
    print(f"Exemples étiquetés : {len(labeled):,} sur {len(base):,}")
    print("Part de stations vides à T+30 :", round(100*labeled["empty_in_30min"].mean(), 2), "%")
    # Baseline naïve (persistance) : vide maintenant => vide dans 30 minutes.
    from sklearn.metrics import accuracy_score, f1_score
    y = labeled["empty_in_30min"].astype(int)
    pred = labeled["empty_now"].astype(int)
    print("Baseline persistance - accuracy :", round(accuracy_score(y, pred), 3))
    print("Baseline persistance - F1 :", round(f1_score(y, pred, zero_division=0), 3))
else:
    print("Il faut davantage d'historique pour construire les cibles T+30 minutes.")


## 7. Interprétation et limites (à compléter en groupe)

- Décrire le nombre d'instantanés, les trous de collecte et les stations indisponibles.
- Présenter au moins deux graphiques commentés et une table de qualité.
- Discuter le décalage entre `snapshot_at_utc` et `last_reported` : une mesure récente du flux n'implique pas que chaque station ait signalé un nouvel état à cet instant.
- Comparer ensuite une éventuelle classification supervisée à la baseline de persistance, uniquement avec un **test chronologique** (aucune fuite de futur).
- Ne pas généraliser sur les saisons ni les comportements hebdomadaires avec 24–48h de données.
- Citer la documentation officielle Vélib', la date et la méthode de collecte.
